In [1]:
import sentencepiece as spm
from sentencepiece import sentencepiece_model_pb2 as spm_proto
import torch
import json

model_path = "/mnt/e/models/unsloth/gemma-3-27b-pt-pruned-vocab"

sp = spm.SentencePieceProcessor(model_file=model_path+"/tokenizer.model")
base_vocab_size = sp.GetPieceSize()
print(f"Base vocab size (from SentencePiece): {base_vocab_size}")  # 262144

# Load added tokens
with open(model_path + "/added_tokens.json", "r") as f:
    added_tokens = json.load(f)
print(f"Added tokens: {added_tokens}")  # {'<image_soft_token>': 262144}

# True vocab size
true_vocab_size = base_vocab_size + len(added_tokens)
print(f"True vocab size (base + added): {true_vocab_size}")  # 262145


Base vocab size (from SentencePiece): 262144
Added tokens: {'<image_soft_token>': 262144}
True vocab size (base + added): 262145


In [2]:
vocab_counts = torch.load(model_path+"/vocab_counts.torch") # vocab_counts[i] = count of token i in the training data
assert len(vocab_counts) == true_vocab_size, "Vocab counts length mismatch"
recur_counts = torch.load(model_path+"/recur_counts.torch") # recur_counts[i] = count of sub-tokens that are encoded as i
assert len(recur_counts) == true_vocab_size, "Recur counts length mismatch"
token_mapping = torch.load(model_path+"/token_mapping.torch").tolist() # token_mapping[i] = token id of the i-th token in the original tokenizer


In [3]:
with open(model_path + "/tokenizer.model", "rb") as f:
    model_proto = spm_proto.ModelProto()
    model_proto.ParseFromString(f.read())


orig_pieces = model_proto.pieces
print(f"Original pieces count: {len(orig_pieces)}")
assert len(orig_pieces) == base_vocab_size, "Original pieces count mismatch"



Original pieces count: 262144


In [5]:
# Add the added token to pieces
for token, old_id in added_tokens.items():
    new_piece = spm_proto.ModelProto.SentencePiece()
    new_piece.piece = token  # '<image_soft_token>'
    new_piece.score = 0.0    # Default score (adjust if needed)
    new_piece.type = spm_proto.ModelProto.SentencePiece.CONTROL  # Special token
    orig_pieces.append(new_piece)

print(f"Updated pieces count with added tokens: {len(orig_pieces)}")  # 262145


Updated pieces count with added tokens: 262145


In [6]:
new_pieces = []
old_to_new = {old_id: new_id for new_id, old_id in enumerate(token_mapping)}  # {0: 0, 1: 1, ..., 262144: 39159}
removed_count = 0
for old_id, piece in enumerate(orig_pieces):
    if old_id in token_mapping:
        new_piece = spm_proto.ModelProto.SentencePiece()
        new_piece.piece = piece.piece
        new_piece.score = piece.score
        new_piece.type = piece.type
        new_pieces.append(new_piece)
    else:
        removed_count += 1

print(f"Removed {removed_count} pieces. Final pieces count: {len(new_pieces)}. Expected count: {len(token_mapping)}")
assert len(new_pieces) == len(token_mapping), "New pieces count mismatch"

Removed 222729 pieces. Final pieces count: 39416. Expected count: 39416


In [7]:
# Ensure new_pieces matches token_mapping order
new_pieces_sorted = [None] * len(token_mapping)
for new_id, old_id in enumerate(token_mapping):
    new_pieces_sorted[new_id] = new_pieces[old_to_new[old_id]]


In [8]:
# Update the model (fix for RepeatedCompositeFieldContainer)
model_proto.pieces.clear()  # Clear existing pieces
model_proto.pieces.extend(new_pieces_sorted)  # Add new pieces
print(f"New pieces count: {len(model_proto.pieces)}")  # 39160
print(f"Removed {removed_count} tokens (expected {true_vocab_size - len(token_mapping)})")  # 222985
assert len(model_proto.pieces) == len(token_mapping), \
    f"New vocab size ({len(model_proto.pieces)}) doesn’t match token_mapping ({len(token_mapping)})"

New pieces count: 39416
Removed 222729 tokens (expected 222729)


In [9]:
# Save the modified model
new_model_path = model_path + "/tokenizer_pruned.model"
with open(new_model_path, "wb") as f:
    f.write(model_proto.SerializeToString())

In [10]:
# Verify the new model
sp_new = spm.SentencePieceProcessor(model_file=model_path+"/tokenizer_pruned.model")
print(f"New vocab size: {sp_new.GetPieceSize()}")  # 39160
for i in range(10):
    token = sp_new.IdToPiece(i)
    print(f"New ID {i}: {token}")

New vocab size: 39416
New ID 0: <pad>
New ID 1: <eos>
New ID 2: <bos>
New ID 3: <unk>
New ID 4: <mask>
New ID 5: [multimodal]
New ID 6: <unused0>
New ID 7: <unused1>
New ID 8: <unused2>
New ID 9: <unused3>
